In [4]:
### Install libraries
!pip install -q chromadb sentence-transformers langgraph fastapi uvicorn pydantic

In [5]:
###import libraries
import os
import chromadb

from sentence_transformers import SentenceTransformer
from typing import TypedDict, List

from pydantic import BaseModel
from fastapi import FastAPI

print("Libraries imported successfully!")

Libraries imported successfully!


In [18]:
import zipfile

with zipfile.ZipFile("/content/docs.zip", "r") as zip_ref:
    zip_ref.extractall("docs")

print("Files extracted successfully!")

Files extracted successfully!


In [22]:
import os

print("Current folder:")
print(os.listdir())

print("\nDocs folder:")
print(os.listdir("docs"))

Current folder:
['.config', 'docs', 'docs.zip', 'sample_data']

Docs folder:
['docs']


In [23]:
import os
print(os.listdir("docs/docs"))

['doc_01.txt', 'doc_05.txt', 'doc_03.txt', 'doc_06.txt', 'doc_02.txt', 'doc_04.txt', 'doc_07.txt', 'doc_08.txt']


In [24]:
##### Load all 8 documents
DOCS_FOLDER = "docs/docs"

documents = []

for file in os.listdir(DOCS_FOLDER):
    if file.endswith(".txt"):
        with open(os.path.join(DOCS_FOLDER, file), "r", encoding="utf-8") as f:
            text = f.read()

        documents.append({
            "id": file,
            "text": text
        })

print(f"Loaded {len(documents)} documents.")

Loaded 8 documents.


In [25]:
# Chunk Documents

chunk_size = 500
chunks = []

for doc in documents:
    text = doc["text"]

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]

        chunks.append({
            "id": f"{doc['id']}_{i // chunk_size}",
            "document": doc["id"],
            "text": chunk
        })

print(f"Total chunks created: {len(chunks)}")

# Display first chunk
print(chunks[0])

Total chunks created: 8
{'id': 'doc_01.txt_0', 'document': 'doc_01.txt', 'text': 'Delivery Policy:\n        Zepto delivers grocery and household essentials to serviceable pin codes within 10–30 minutes.\n        Standard delivery is free on orders above INR 149.\n        Orders below INR 149 incur a flat INR 25 delivery fee.\n        Priority delivery costs an additional INR 15.\n        Zepto does not deliver outside serviceable pin codes.'}


In [26]:
#  Generate Embeddings

from sentence_transformers import SentenceTransformer

print("Loading embedding model...")

model = SentenceTransformer("all-MiniLM-L6-v2")

for chunk in chunks:
    chunk["embedding"] = model.encode(chunk["text"]).tolist()

print(f"Generated embeddings for {len(chunks)} chunks.")

# Check one embedding
print("Embedding length:", len(chunks[0]["embedding"]))

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generated embeddings for 8 chunks.
Embedding length: 384


In [27]:
# Cell 6: Store Embeddings in ChromaDB

import chromadb

# Create persistent ChromaDB client
client = chromadb.PersistentClient(path="chroma_db")

# Create or get collection
collection = client.get_or_create_collection(name="zepto_policy")

# Add all chunks
for chunk in chunks:
    collection.add(
        ids=[chunk["id"]],
        documents=[chunk["text"]],
        embeddings=[chunk["embedding"]],
        metadatas=[{"document": chunk["document"]}]
    )

print(f"Stored {len(chunks)} chunks in ChromaDB.")

# Verify
print("Total records in ChromaDB:", collection.count())

Stored 8 chunks in ChromaDB.
Total records in ChromaDB: 8


In [28]:
#  Structured Prompt Template

PROMPT_TEMPLATE = """
ROLE:
You are a helpful Zepto customer support assistant.

CONTEXT:
{context}

TASK:
Answer the user's question using ONLY the information provided in the context.

FORMAT:
Return a clear and concise answer in plain English.

LENGTH:
Maximum 150 words.

NEGATIVE CONSTRAINT:
Do not answer using information that is NOT present in the provided context.
If the answer is not available in the context, say:
"I could not find this information in the provided policy documents."

FEW-SHOT EXAMPLE:

Example 1

Question:
How can I cancel my order?

Answer:
According to the provided policy, orders can be cancelled before dispatch if they meet the cancellation conditions mentioned in the policy.

NOW ANSWER:

Question:
{question}

Answer:
"""

print("Structured prompt template created successfully!")

Structured prompt template created successfully!


In [29]:
### Pydantic Models

from pydantic import BaseModel
from typing import List

class AskRequest(BaseModel):
    query: str

class AskResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float

print("Pydantic models created successfully!")

Pydantic models created successfully!


In [30]:
#  LangGraph State

from typing import TypedDict, List

class GraphState(TypedDict):
    query: str
    intent: str
    answer: str
    sources: List[str]
    confidence: float

print("GraphState created successfully!")

GraphState created successfully!


In [31]:
### MOCK_LLM Configuration

import os

# Default is mock mode (required for grading)
MOCK_LLM = os.getenv("MOCK_LLM", "1") == "1"

print("MOCK_LLM:", MOCK_LLM)

MOCK_LLM: True


In [32]:
# Cell 11: classify_intent()

keywords = [
    "delivery",
    "return",
    "refund",
    "membership",
    "tracking",
    "cancel",
    "gift card",
    "support hours"
]

def classify_intent(state: GraphState):
    query = state["query"].lower()

    if MOCK_LLM:
        if any(keyword in query for keyword in keywords):
            intent = "policy_question"
        else:
            intent = "general_question"
    else:
        # Optional real LLM implementation
        intent = "policy_question"

    state["intent"] = intent
    return state

print("classify_intent node created successfully!")

classify_intent node created successfully!


In [33]:
#  retrieve_and_answer()

def retrieve_and_answer(state: GraphState):

    query = state["query"]

    # Embed the query
    query_embedding = model.encode(query).tolist()

    # Retrieve top 3 chunks
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3
    )

    top_chunk = results["documents"][0][0]
    source_ids = results["ids"][0]

    if MOCK_LLM:
        answer = f"Based on the retrieved context: {top_chunk[:200]}"
    else:
        # Optional real LLM
        answer = "Real LLM answer goes here."

    state["answer"] = answer
    state["sources"] = source_ids
    state["confidence"] = 1.0

    return state

print("retrieve_and_answer node created successfully!")

retrieve_and_answer node created successfully!


In [34]:
# direct_answer()

def direct_answer(state: GraphState):

    if MOCK_LLM:
        answer = "I can only answer questions about Zepto policies right now."
    else:
        # Optional real LLM
        answer = "Real LLM answer."

    state["answer"] = answer
    state["sources"] = []
    state["confidence"] = 1.0

    return state

print("direct_answer node created successfully!")

direct_answer node created successfully!


In [35]:
#  Build LangGraph

from langgraph.graph import StateGraph, START, END

# Create graph
builder = StateGraph(GraphState)

# Add nodes
builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_and_answer", retrieve_and_answer)
builder.add_node("direct_answer", direct_answer)

# Start node
builder.add_edge(START, "classify_intent")

# Conditional routing
def route(state: GraphState):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    else:
        return "direct_answer"

builder.add_conditional_edges(
    "classify_intent",
    route,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer",
    }
)

# End nodes
builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

# Compile graph
graph = builder.compile()

print("LangGraph compiled successfully!")

LangGraph compiled successfully!


In [36]:
#  Test the graph

result = graph.invoke({
    "query": "How can I cancel my order?"
})

print(result)

{'query': 'How can I cancel my order?', 'intent': 'policy_question', 'answer': 'Based on the retrieved context: Order Cancellation:\n        Orders can be cancelled free before they are packed.\n        Packed orders cannot be cancelled through the app.\n        Zepto-side issues result in automatic cancellation a', 'sources': ['doc_05.txt_0', 'doc_02.txt_0', 'doc_01.txt_0'], 'confidence': 1.0}


In [37]:
#  FastAPI App

from fastapi import FastAPI

app = FastAPI(title="Zepto Policy RAG API")

@app.post("/ask", response_model=AskResponse)
def ask(request: AskRequest):

    state = {
        "query": request.query,
        "intent": "",
        "answer": "",
        "sources": [],
        "confidence": 0.0
    }

    result = graph.invoke(state)

    return AskResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"]
    )

print("FastAPI app created successfully!")

FastAPI app created successfully!


In [38]:
#### Test Example
result = graph.invoke({
    "query": "How do I cancel my order?",
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
})

print(result)

{'query': 'How do I cancel my order?', 'intent': 'policy_question', 'answer': 'Based on the retrieved context: Order Cancellation:\n        Orders can be cancelled free before they are packed.\n        Packed orders cannot be cancelled through the app.\n        Zepto-side issues result in automatic cancellation a', 'sources': ['doc_05.txt_0', 'doc_02.txt_0', 'doc_01.txt_0'], 'confidence': 1.0}


In [39]:
result = graph.invoke({
    "query": "Who is the Prime Minister of India?",
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
})

print(result)

{'query': 'Who is the Prime Minister of India?', 'intent': 'general_question', 'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}
